In [7]:
#installing important packages
conda install numpy opencv matplotlib

3 channel Terms of Service accepted
Channels:
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\kaykv\anaconda3\envs\Personal_projects

  added / updated specs:
    - matplotlib
    - numpy
    - opencv


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    aws-c-auth-0.9.4           |       h02ab6af_0         118 KB
    aws-c-cal-0.9.13           |       h630b2a1_0          58 KB
    aws-c-common-0.12.6        |       h02ab6af_0         238 KB
    aws-c-compression-0.3.1    |       h02ab6af_3          29 KB
    aws-c-http-0.10.7          |       h02ab6af_0         193 KB
    aws-c-io-0.23.3            |       h02ab6af_0         170 KB
    aws-c-s3-0.11.3            |       h02ab6af_0         132 KB
    aws-c-sdkutils-0.2.4       |       h02ab6af_2          60 KB
    aws-checksums-0.2.8        |       h02ab6af_0         105 KB
    exp



==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda




In [ ]:
pip install mediapipe # media pipe is basically google's computer vision based API package, A.K.A thank you Mr megacorporation engineering team

In [18]:
# Importing the packages in helpfyul formats


import numpy as np
import cv2
import mediapipe as mp 
import time


In [ ]:
'''Note - There are some very obvious redundancy checks that I didn't put in  partially because those case scenarios are very hard to come across in this scenario, 
if this code was implemented in the field its very obvious that the implementation would be stronger( this is just an MVP)'''

In [25]:
def draw_landmarks_on_stream(rgb_frame, detection_result):
    annotated_frame = np.copy(rgb_frame)
    height, width, _ = annotated_frame.shape

    for face_landmarks in detection_result.face_landmarks:
        for landmark in face_landmarks:
            x = int(landmark.x * width)
            y = int(landmark.y * height)
            cv2.circle(annotated_frame, (x, y), 1, (0, 255, 0), -1)  #draw each landmark dot


    return annotated_frame

In [ ]:
# so the idea is that capturing frames in a loop will yield in a video 
# Our main function that defines our camera source and purpose
capture = cv2.VideoCapture(0) 

# Importing the task file (pretrained)
model_path = 'face_landmarker.task'

# defining the function that's gonna help us draw 



# Importing the necessary classes 
''' AI generated description : Think of them as a small assembly line:

BaseOptions → tells the detector where the model is and which device to use.
FaceLandmarkerOptions → configures the detector using BaseOptions plus other parameters (faces, confidence, running mode).
VisionRunningMode → tells the detector how to process frames (single image, video, live).
FaceLandmarker → the detector object you create using FaceLandmarkerOptions.
FaceLandmarkerResult → what you get after calling .process(frame) on FaceLandmarker.'''

BaseOptions = mp.tasks.BaseOptions 
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
FaceLandmarkerResult = mp.tasks.vision.FaceLandmarkerResult
VisionRunningMode = mp.tasks.vision.RunningMode


# defining the basics such as our path for the model we are going to use
base_options = BaseOptions(
    model_asset_path="face_landmarker.task"
) 

annotated_stream = None
#defining a final callback function to use in our code under options
def trace_result_face(result: FaceLandmarkerResult,output_image: mp.Image,timestamp_ms: int):
    global annotated_stream
    rgb_frame = output_image.numpy_view()
    annotated_stream = draw_landmarks_on_stream(rgb_frame, result)

    
# defining extra features such as our running mode and feeding our base options class further into a "pipeline"( of sorts)
options = FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=VisionRunningMode.LIVE_STREAM, # utilising the proper running mode for our data 
    num_faces = 1,
    min_face_detection_confidence = 0.5,
    min_face_presence_confidence = 0.5,
    result_callback = trace_result_face)

# creating an actual landmarker object 
landmarker = FaceLandmarker.create_from_options(options)

    
# The main frame loop
while True : 

    frame_timestamp_ms = int(time.time() * 1000) # defining our real time 

    checking_bool, frame = capture.read() # ret is a boolean value to see if the frame was capture successfully and frame is the actual image capture

    if not checking_bool:
        print("Not able to capture the frame successfully")
        break

    rgb_frame = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB) # changing the colour of our input from BGR to RGB cause mediapipe takes RGB

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)# Convert the frame received from OpenCV to a MediaPipe’s Image object.

    result = landmarker.detect_async(mp_image, frame_timestamp_ms)

    if annotated_stream is not None:
        bgr_output = cv2.cvtColor(annotated_stream, cv2.COLOR_RGB2BGR)  # convert back to BGR 
        cv2.imshow('Face Landmarks', bgr_output)
    else:
        cv2.imshow('Face Landmarks', frame)  # show plain frame until callback fires
    

   

    if cv2.waitKey(33) == ord('c') : # wait for this much time in milliseconds before each frame , if c is pressed then releases the video and closes all windows, aiming for 30 fps
        break

landmarker.close()
capture.release()
cv2.destroyAllWindows() # A very dramatic sounding function to close the captured windows






In [ ]:
# I tried my best to go through the documentation as much as I could and use AI as less as possible to make sure I balance my learning and implementation skills


# AI usage - https://chatgpt.com/share/69ce5a9a-7504-8321-aeb8-baacdf0182a6 ,https://chatgpt.com/c/69d0bc04-fbb4-8322-8264-d83d3f3c7dd1, https://claude.ai/chat/eb235ade-5185-4ae5-9550-449da1ef54dc 

''' Internet resources used :
https://docs.opencv.org/4.13.0/
https://medium.com/@alionurulker/live-stream-on-any-camera-using-opencv-and-python-e18d4de6fad7 
https://docs.opencv.org/4.x/dd/d43/tutorial_py_video_display.html 
https://youtu.be/RnQy7U5Eu94?si=06W-LhxrypmYoJu5
https://www.geeksforgeeks.org/python/python-opencv-waitkey-function 
https://youtu.be/hV5S4iQhNkI?si=aZB8sLVnCiwGMbkV 
https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/python 
https://youtu.be/rAS17tDYeA0?si=pU7OKbfWT9PtvrfB # ditched this resource because it was build off of a legacy API but utilised it regardless
https://youtu.be/loEZnF7Z-Zk?si=VDWIOGi6ZLPP0kdS 


'''